In [ ]:
incdnt_sid = 'EVE-5000001'


In [ ]:
import datetime
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.enableHiveSupport().getOrCreate()
spark.conf.set('spark.sql.execution.arrow.pyspark.enabled', 'true')

DM_NAME = 'arnsdpsbx_t_team_sva_oarb_4.'
MAIN_TABLE = 'd6_base_of_knowledge_ior'

main_df = spark.table(DM_NAME + MAIN_TABLE).filter(
    F.col('incdnt_sid') == incdnt_sid
)
fin_df = spark.table(DM_NAME + 'd6_base_of_knowledge_incident_fin_impact')
rec_df = spark.table(DM_NAME + 'd6_base_of_knowledge_incident_recovery')
nonfin_df = spark.table(DM_NAME + 'd6_base_of_knowledge_incident_nonfin_impact')

report = main_df.join(fin_df, 'incdnt_id', 'left') \
                .join(rec_df, 'incdnt_id', 'left') \
                .join(nonfin_df, 'incdnt_id', 'left')

ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
file_name = f"Отчёт по ИОР {incdnt_sid}_{ts}.xlsx"
df = report.toPandas()
df.columns = [c.lower() for c in df.columns]

RENAME = {
    'incdnt_id': 'Идентификационный ключ инцидента операционного риска',
    'incdnt_sid': 'Идентификатор события',
    'incdnt_status_name': 'Статус события',
    'incdnt_autoreg_flag': 'Признак авторегистрации инцидента',
    'incdnt_detection_person_name': 'Кем выявлено событие',
    'incdnt_source_name': 'Название источника',
    'src_type_lvl_1_name': 'Тип источника инцидента (уровень 1)',
    'src_type_lvl_2_name': 'Тип источника инцидента (уровень 2)',
    'incdnt_type_lvl_1_name': 'Тип события – уровень 1',
    'incdnt_type_lvl_2_name': 'Тип события – уровень 2',
    'incdnt_detection_dt': 'Дата обнаружения (Событие)',
    'incdnt_start_dt': 'Дата начала инцидента операционного риска',
    'incdnt_entry_dt': 'Дата ввода (Событие)',
    'org_struct_lvl_2_name': 'Орг. структура – уровень 2 (Терр. структура / Департамент ДЗО)',
    'org_struct_lvl_3_name': 'Орг. структура – уровень 3 (Блок / ТБ / ПЦП)',
    'org_struct_lvl_4_name': 'Орг. структура – уровень 4 (Дивизион / Департамент)',
    'process_lvl_4_name': 'Процесс – уровень 4 (Наименование процесса)',
    'incdnt_summary_descr_txt': 'Предварительное описание',
    'incdnt_full_descr_txt': 'Подробное описание',
    'incdnt_sum': 'Общая сумма всех последствий (руб.)',
    'incdnt_drct_dmg_sum': 'Прямая потеря – итого (руб.)',
    'recovery_type_name': 'Тип возмещения',
    'recovery_rub_amt': 'Сумма возмещения, ₽',
    'fin_impact_type_name': 'Тип последствия',
    'fin_impact_rub_amt': 'Сумма последствия, ₽',
    'nonfin_impact_kind_name': 'Вид качественной потери',
    'nonfin_impact_influence_class_name': 'Класс влияния'
}

df = df.rename(columns=RENAME)
df.to_excel(file_name, sheet_name='Отчет_ОпРиски', index=False, engine='openpyxl')
spark.stop()
